# 持久性
Langgraph具有通过检查器实现的内置持久性层。 当您使用CheckPointer编译图形时，CheckPointer在每个超级步骤上保存图形状态的`checkpoint`。 这些`checkpoint`被保存到`thread`，可以在图形执行后访问。 由于线程在执行后允许访问Graph的状态，因此可以进行几种强大的功能，包括人机交互、内存、时间旅行和容错。 下面，我们将更详细地讨论这些概念。

<img src="https://langchain-ai.github.io/langgraph/concepts/img/persistence/checkpoints.jpg">

## 线程¶
线程是检查点器保存的每个检查点分配的唯一 ID 或线程标识符。它包含一系列运行的累积状态。执行运行后，助手底层图的状态将持久保存到线程中。

用检查点调用图表时，必须指定`thread_id`作为`configurable`配置部分的一部分：

```json
{"configurable": {"thread_id": "1"}}
```

可以检索线程的当前和历史状态。为了持久化状态，必须在执行运行之前创建一个线程。LangGraph 平台 API 提供了多个用于创建和管理线程及线程状态的端点。更多详情，请[参阅API 参考](https://langchain-ai.github.io/langgraph/cloud/reference/api/api_ref.html#tag/threads)。

## 检查点 
线程在特定时间点的状态称为检查点。检查点是每个超级步骤保存的图形状态快照，由具有以下关键属性的 StateSnapshot 对象表示：
- config：与此检查点相关的配置。
- metadata：与此检查点相关的元数据。
- values：此时状态通道的值。
- next图中接下来要执行的节点名称的元组。
- tasksPregelTask：包含后续待执行任务信息的对象元组。如果之前尝试过此步骤，则将包含错误信息。如果图在节点内部被动态中断，则任务将包含与中断相关的其他数据。

检查点是持久的，可用于稍后恢复线程的状态。

让我们看看当调用一个简单的图表时会保存哪些检查点，如下所示：

API 参考：StateGraph |开始|结束| InMemorySaver

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "1"}}
graph.invoke({"foo": ""}, config)

{'foo': 'b', 'bar': ['a', 'b']}

运行图表后，我们预计会看到 4 个检查点：

- 空检查点，START作为下一个要执行的节点
- 检查点与用户输入{'foo': '', 'bar': []}并`node_a`作为下一个要执行的节点
- `node_a`检查点输出为 {'foo': 'a', 'bar': ['a']}和`node_b`作为下一个要执行的节点
- `node_b`检查点输出为 {'foo': 'b', 'bar': ['a', 'b']}并且没有要执行的下一个节点

请注意，bar通道值包含两个节点的输出，因为我们为条形通道设置了一个reducer函数（add）。



### 获取状态
与保存的图形状态交互时，必须指定[线程标识符](https://langchain-ai.github.io/langgraph/concepts/persistence/#threads)。您可以调用 `graph.get_state(config) `查看图形的最新状态。这将返回一个 `StateSnapshot` 对象，该对象对应于与配置中提供的线程 ID 相关联的最新检查点，或者与线程的检查点 ID 相关联的检查点（如果提供）。



In [ ]:
# get the latest state snapshot
config = {"configurable": {"thread_id": "1"}}
graph.get_state(config)

# get a state snapshot for a specific checkpoint_id
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1ef663ba-28fe-6528-8002-5a559208592c"}}
graph.get_state(config)

在我们的示例中，输出`get_state`将如下所示：

```shell
StateSnapshot(
    values={'foo': 'b', 'bar': ['a', 'b']},
    next=(),
    config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28fe-6528-8002-5a559208592c'}},
    metadata={'source': 'loop', 'writes': {'node_b': {'foo': 'b', 'bar': ['b']}}, 'step': 2},
    created_at='2024-08-29T19:19:38.821749+00:00',
    parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f9-6ec4-8001-31981c2c39f8'}}, tasks=()
)
```

通过调用 `graph.get_state_history(config)`，可以获取给定线程的完整图形执行历史。这将返回与配置中提供的线程 ID 相关联的状态快照对象列表。重要的是，检查点将按时间顺序排列，最近的检查点/状态快照将排在列表的第一位。

In [ ]:
config = {"configurable": {"thread_id": "1"}}
list(graph.get_state_history(config))

在我们的示例中，`get_state_history` 的输出将如下所示：

```shell
[
    StateSnapshot(
        values={'foo': 'b', 'bar': ['a', 'b']},
        next=(),
        config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28fe-6528-8002-5a559208592c'}},
        metadata={'source': 'loop', 'writes': {'node_b': {'foo': 'b', 'bar': ['b']}}, 'step': 2},
        created_at='2024-08-29T19:19:38.821749+00:00',
        parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f9-6ec4-8001-31981c2c39f8'}},
        tasks=(),
    ),
    StateSnapshot(
        values={'foo': 'a', 'bar': ['a']}, next=('node_b',),
        config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f9-6ec4-8001-31981c2c39f8'}},
        metadata={'source': 'loop', 'writes': {'node_a': {'foo': 'a', 'bar': ['a']}}, 'step': 1},
        created_at='2024-08-29T19:19:38.819946+00:00',
        parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f4-6b4a-8000-ca575a13d36a'}},
        tasks=(PregelTask(id='6fb7314f-f114-5413-a1f3-d37dfe98ff44', name='node_b', error=None, interrupts=()),),
    ),
    StateSnapshot(
        values={'foo': '', 'bar': []},
        next=('node_a',),
        config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f4-6b4a-8000-ca575a13d36a'}},
        metadata={'source': 'loop', 'writes': None, 'step': 0},
        created_at='2024-08-29T19:19:38.817813+00:00',
        parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f0-6c66-bfff-6723431e8481'}},
        tasks=(PregelTask(id='f1b14528-5ee5-579c-949b-23ef9bfbed58', name='node_a', error=None, interrupts=()),),
    ),
    StateSnapshot(
        values={'bar': []},
        next=('__start__',),
        config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f0-6c66-bfff-6723431e8481'}},
        metadata={'source': 'input', 'writes': {'foo': ''}, 'step': -1},
        created_at='2024-08-29T19:19:38.816205+00:00',
        parent_config=None,
        tasks=(PregelTask(id='6d27aa2e-d72b-5504-a36f-8620e54a76dd', name='__start__', error=None, interrupts=()),),
    )
]
```


<img src="https://langchain-ai.github.io/langgraph/concepts/img/persistence/get_state.jpg">

### 回放 
还可以回放之前的图形执行。如果我们调用一个带有`thread_id` 和`checkpoint_id` 的图，那么我们将重放与`checkpoint_id` 相对应的检查点之前已执行的步骤，并只执行检查点之后的步骤。

- thread_id是线程的ID。
- checkpoint_id是指代线程内特定检查点的标识符。

在调用图表作为`configurable`配置部分的一部分时，必须传递这些内容：



In [ ]:
config = {"configurable": {"thread_id": "1", "checkpoint_id": "0c62ca34-ac19-445d-bbb0-5b4984975b2a"}}
graph.invoke(None, config=config)

重要的是，LangGraph 知道某个步骤之前是否已经执行过。如果已经执行过，LangGraph 只需在图中重新显示该特定步骤，而不会重新执行该步骤，但只会执行所提供的 `checkpoint_id` 之前的步骤。`checkpoint_id` 之后的所有步骤都将被执行（即一个新的分叉），即使它们之前已经执行过。有关重放的更多信息，请参阅时间旅行指南。

<img src="https://langchain-ai.github.io/langgraph/concepts/img/persistence/re_play.png">

### 更新状态
除了从特定检查点重放图表外，我们还可以编辑图表状态。我们使用 `graph.update_state（） `来做到这一点。此方法接受三个不同的参数：

#### config
配置应该包含`thread_id`指定要更新哪个线程的信息。当仅thread_id传递 时，我们会更新（或分叉）当前状态。（可选）如果我们包含`checkpoint_id`字段，则我们会分叉所选的检查点。

#### values

这些值将用于更新状态。请注意，此更新的处理方式与处理来自节点的任何更新完全相同。这意味着如果这些值是为图状态中的某些通道定义的，则这些值将传递给 `reducer` 函数。这意味着`update_state`不会自动覆盖每个通道的通道值，而只会自动覆盖没有减速器的通道值。让我们看一个例子。

假设您已使用以下模式定义了图表的状态（请参阅上面的完整示例）：

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: int
    bar: Annotated[list[str], add]

如果您按如下方式更新状态：

```shell
graph.update_state(config, {"foo": 2, "bar": ["b"]})
```

键foo（通道）已完全更改（因为该通道未指定` Reducer`，因此`update_state`会覆盖它）。但是，该键bar已指定Reducer ，因此它会附加"b"到 bar的状态。

### as_node
调用时，最后一个可选参数`update_state`是`as_node`。如果提供了 ，更新将被视为来自节点`as_node`。如果`as_node`没有提供 ，则在不产生歧义的情况下，它将设置为最后一个更新状态的节点。之所以如此重要，是因为接下来要执行的步骤取决于最后一个提供更新的节点，因此可以使用它来控制接下来执行哪个节点。请参阅这篇关于时间旅行的操作指南，了解更多关于分叉状态 的信息。

<img src="https://langchain-ai.github.io/langgraph/concepts/img/persistence/checkpoints_full_story.jpg">

## 记忆存储

<img scr="https://langchain-ai.github.io/langgraph/concepts/img/persistence/shared_state.png">

状态模式指定了一组在执行图时填充的键。如上所述，可以通过检查点在执行图的每个步骤将状态写入线程，从而实现状态持久化。

但是，如果我们想在各个线程之间保留一些信息怎么办？考虑一个聊天机器人的情况，我们希望在与该用户的所有聊天对话（例如，线程）中保留有关该用户的特定信息！

仅靠校验器，我们无法跨线程共享信息。这就促使我们需要存储接口。举例来说，我们可以定义一个 `InMemoryStore` 来跨线程存储用户信息。我们只需像以前一样，使用校验指针和新的 `in_memory_store` 变量编译我们的图。

> 使用 LangGraph API 时，您无需手动实现或配置存储。API 会在后台为您处理所有存储基础设施。

## 基本用法¶
首先，让我们在不使用 LangGraph 的情况下单独展示这一点。


In [ ]:
from langgraph.store.memory import InMemoryStore
in_memory_store = InMemoryStore()

记忆的命名空间为`tuple`，在本例中为`(<user_id>, "memories")`。命名空间可以是任意长度，可以表示任何内容，不必特定于用户。

In [ ]:
user_id = "1"
namespace_for_memory = (user_id, "memories")

我们使用此store.put方法将记忆保存到存储中的命名空间中。执行此操作时，我们指定上面定义的命名空间，以及记忆的键值对：键只是记忆的唯一标识符（memory_id），而值（字典）是记忆本身。

In [ ]:
memory_id = str(uuid.uuid4())
memory = {"food_preference" : "I like pizza"}
in_memory_store.put(namespace_for_memory, memory_id, memory)

我们可以使用 `store.search` 方法读取命名空间中的记忆，该方法会以列表形式返回给定用户的所有记忆。最近的记忆是列表中的最后一个。

In [ ]:
memories = in_memory_store.search(namespace_for_memory)
memories[-1].dict()
{'value': {'food_preference': 'I like pizza'},
 'key': '07e0caf4-1631-47b7-b15f-65515d4c1843',
 'namespace': ['1', 'memories'],
 'created_at': '2024-10-02T17:22:31.590602+00:00',
 'updated_at': '2024-10-02T17:22:31.590605+00:00'}

每种内存类型都是一个具有特定属性的 `Python 类（Item）`。我们可以通过上述的 `.dict` 转换将其作为字典访问。其属性如下

- value：此内存的值（本身就是一个字典）
- key：此命名空间中此内存的唯一键
- namespace：字符串列表，此内存类型的命名空间
- created_at：此内存创建的时间戳
- updated_at：此内存更新的时间戳

### 语义搜索¶
除了简单的检索功能外，存储还支持语义搜索，让您能够根据含义而非精确匹配来查找记忆。要启用此功能，请在存储中配置嵌入模型：

API 参考：[init_embeddings](https://python.langchain.com/api_reference/langchain/embeddings/langchain.embeddings.base.init_embeddings.html?_gl=1*dy0jnk*_gcl_au*NDE1NjY4Mjc0LjE3NTM0Mjc3MTc.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NTM4Mzc1NTUkbzE3JGcxJHQxNzUzODM3NjUyJGo2MCRsMCRoMA..)

In [ ]:
from langchain.embeddings import init_embeddings

store = InMemoryStore(
    index={
        "embed": init_embeddings("openai:text-embedding-3-small"),  # Embedding provider
        "dims": 1536,                              # Embedding dimensions
        "fields": ["food_preference", "$"]              # Fields to embed
    }
)

现在搜索时，您可以使用自然语言查询来查找相关的记忆：

In [ ]:
# Find memories about food preferences
# (This can be done after putting memories into the store)
memories = store.search(
    namespace_for_memory,
    query="What does the user like to eat?",
    limit=3  # Return top 3 matches
)

您可以通过配置fields参数或在存储记忆时指定index参数来控制嵌入记忆的部分：

In [ ]:
from langchain.embeddings import init_embeddings

store = InMemoryStore(
    index={
        "embed": init_embeddings("openai:text-embedding-3-small"),  # Embedding provider
        "dims": 1536,                              # Embedding dimensions
        "fields": ["food_preference", "$"]              # Fields to embed
    }
)

### 在 LangGraph 中使用
有了这一切，我们就可以在 LangGraph 中使用内存存储（`in_memory_store`）了。内存中存储（`in_memory_store`）与校验指针（`checkpointer`）协同工作：如上所述，校验指针会将状态保存到线程中，而内存中存储（`in_memory_store`）允许我们存储任意信息，以便跨线程访问。我们使用 `checkpointer` 和 `in_memory_store` 对图进行如下编译。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# We need this because we want to enable threads (conversations)
checkpointer = InMemorySaver()

# ... Define the graph ...

# Compile the graph with the checkpointer and store
graph = graph.compile(checkpointer=checkpointer, store=in_memory_store)

像以前一样，我们使用 thread_id来调用图表，同样使用 user_id来将我们的记忆命名到这个特定的用户，正如我们上面所展示的那样。

In [ ]:
# Invoke the graph
user_id = "1"
config = {"configurable": {"thread_id": "1", "user_id": user_id}}

# First let's just say hi to the AI
for update in graph.stream(
    {"messages": [{"role": "user", "content": "hi"}]}, config, stream_mode="updates"
):
    print(update)

通过传递 `store.BaseStore` 和 `config`，我们可以访问任何节点中的 `in_memory_store` 和 `user_id：BaseStore` 和 `config：RunnableConfig` 作为节点参数。下面是我们在节点中使用语义搜索查找相关内存的方法：

In [ ]:
def update_memory(state: MessagesState, 
                  config: RunnableConfig, *, 
                  store: BaseStore):

    # Get the user id from the config
    user_id = config["configurable"]["user_id"]

    # Namespace the memory
    namespace = (user_id, "memories")

    # ... Analyze conversation and create a new memory

    # Create a new memory ID
    memory_id = str(uuid.uuid4())

    # We create a new memory
    store.put(namespace, memory_id, {"memory": memory})

正如我们上面所展示的，我们也可以访问任意节点中的存储，并使用该`store.search`方法来获取记忆。回想一下，这些记忆以对象列表的形式返回，这些对象可以转换为字典。

我们可以访问这些记忆并在我们的模型调用中使用它们。

In [ ]:
def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    # Get the user id from the config
    user_id = config["configurable"]["user_id"]

    # Namespace the memory
    namespace = (user_id, "memories")

    # Search based on the most recent message
    memories = store.search(
        namespace,
        query=state["messages"][-1].content,
        limit=3
    )
    info = "\n".join([d.value["memory"] for d in memories])

    # ... Use memories in the model call

如果我们创建了一个新线程，只要user_id 相同，我们仍然可以访问相同的内存。

In [ ]:
# Invoke the graph
config = {"configurable": {"thread_id": "2", "user_id": "1"}}

# Let's say hi again
for update in graph.stream(
    {"messages": [{"role": "user", "content": "hi, tell me about my memories"}]}, config, stream_mode="updates"
):
    print(update)

当我们使用LangGraph平台时，无论是在本地（例如在LangGraph Studio中）还是在LangGraph平台上，默认情况下都可以使用基础存储，无需在图形编译期间指定。不过，要启用语义搜索，您确实需要在 `langgraph.json` 文件中配置索引设置。例如

In [ ]:
{
    ...
    "store": {
        "index": {
            "embed": "openai:text-embeddings-3-small",
            "dims": 1536,
            "fields": ["$"]
        }
    }
}

有关更多详细信息和配置选项，请参[阅部署指南](https://langchain-ai.github.io/langgraph/cloud/deployment/semantic_search/)。



## 检查点库¶
在底层，检查点由符合BaseCheckpointSaver接口的检查点对象驱动。LangGraph 提供了几种检查点实现，均通过独立的可安装库实现：

- langgraph-checkpoint：检查点保存器（ BaseCheckpointSaver）和序列化/反序列化接口（SerializerProtocol ）的基接口。包含用于实验的内存检查点实现（InMemorySaver）。LangGraph 已包含langgraph-checkpoint在内。
- langgraph-checkpoint-sqlite：LangGraph 检查点的实现，使用 SQLite 数据库（SqliteSaver / AsyncSqliteSaver）。非常适合实验和本地工作流。需要单独安装。
- langgraph-checkpoint-postgres：LangGraph 平台使用 Postgres 数据库（ PostgresSaver / AsyncPostgresSaver ）的高级检查点程序。非常适合在生产环境中使用。需要单独安装。

### 检查点接口¶
每个检查点都符合`BaseCheckpointSaver`接口并实现以下方法：

- `.put` - 存储检查点及其配置和元数据。 
- `.put_writes` - 存储与检查点相关联的中间写入（即待处理写入）。 
- `.get_tuple` - 使用给定配置（`thread_id` 和`checkpoint_id`）获取检查点元组。这将用于填充 `graph.get_state()` 中的 `StateSnapshot`。 
- `.list` - 列出符合给定配置和筛选条件的检查点。用于在 `graph.get_state_history()` 中填充状态历史。

如果`CheckPointer`与异步图执行一起使用（即通过`.AINVOKE`，`.ASTREAM`，`.abatch`执行图），则将使用上述方法的异步版本（`.aput`，`.aput_writes`，`.aput_writes`，`.aget_tuple`，`.aget_tuple`，`.alist`

### 序列化器¶
当检查点保存图状态时，它们需要序列化状态中的通道值。这可以通过序列化器对象完成。 langgraph_checkpoint定义了用于实现序列化器的协议，并提供了一个默认实现 ( JsonPlusSerializer )，可以处理各种类型，包括 LangChain 和 LangGraph 原语、日期时间、枚举等等。

### 使用 pickle¶ 
进行序列化 默认序列化器 `JsonPlusSerializer` 在引擎盖下使用 ormsgpack 和 JSON，但并不适用于所有类型的对象。



In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer

# ... Define the graph ...
graph.compile(
    checkpointer=InMemorySaver(serde=JsonPlusSerializer(pickle_fallback=True))
)

### 加密
检查点程序可以选择加密所有持久化状态。要启用此功能，请在任何 BaseCheckpointSaver 实现的 serde 参数中传递一个加密序列化器（EncryptedSerializer）实例。创建加密序列器的最简单方法是通过 from_pycryptodome_aes，它可以从 LANGGRAPH_AES_KEY 环境变量读取 AES 密钥（或接受一个密钥参数）：

In [ ]:
import sqlite3

from langgraph.checkpoint.serde.encrypted import EncryptedSerializer
from langgraph.checkpoint.sqlite import SqliteSaver

serde = EncryptedSerializer.from_pycryptodome_aes()  # reads LANGGRAPH_AES_KEY
checkpointer = SqliteSaver(sqlite3.connect("checkpoint.db"), serde=serde)

In [ ]:
from langgraph.checkpoint.serde.encrypted import EncryptedSerializer
from langgraph.checkpoint.postgres import PostgresSaver

serde = EncryptedSerializer.from_pycryptodome_aes()
checkpointer = PostgresSaver.from_conn_string("postgresql://...", serde=serde)
checkpointer.setup()

在 LangGraph Platform 上运行时，只要存在 LANGGRAPH_AES_KEY，加密功能就会自动启用，因此只需提供环境变量即可。通过实施 CipherProtocol 并将其提供给 EncryptedSerializer，还可以使用其他加密方案。

## 功能¶
### 人机交互¶
首先，检查点允许人类检查、中断和批准图步骤，从而促进人机交互工作流程。这些工作流程需要检查点，因为人类必须能够在任何时间点查看图的状态，并且图必须在人类对状态进行任何更新后恢复执行。请参阅操作指南中的示例。

### 记忆¶
其次，检查点允许在交互之间进行“记忆”。对于重复的人机交互（例如对话），任何后续消息都可以发送到该线程，该线程将保留先前消息的记忆。有关如何使用检查点添加和管理对话记忆的信息，请参阅添加记忆。

### 时间旅行¶
第三，检查点支持“时间旅行”，允许用户重放之前的图执行，以审查和/或调试特定的图步骤。此外，检查点还可以在任意检查点处分叉图状态，以探索其他路径。

### 容错¶
最后，检查点还提供容错和错误恢复功能：如果一个或多个节点在给定的超级步骤中失败，您可以从上一个成功的步骤重新启动图。此外，当图节点在给定的超级步骤中执行失败时，LangGraph 会存储来自在该超级步骤中成功完成的任何其他节点的待处理检查点写入，这样，每当我们从该超级步骤恢复图执行时，我们都不会重新运行成功的节点。

### 待处理的写入¶
此外，当图形节点在给定的超级步骤中执行过程中失败时，LangGraph 会存储在该超级步骤中成功完成的任何其他节点的待处理检查点写入，以便每当我们从该超级步骤恢复图形执行时，我们都不会重新运行成功的节点。